In [1]:
# Day 4: SQL Analysis Queries
# Goal: Answer business questions using SQL

import pandas as pd
import sqlite3

# Connect to database
conn = sqlite3.connect('complaints_analysis.db')
print("="*60)
print("SQL ANALYSIS: BUSINESS QUESTIONS")
print("="*60)

# ============================================================
# QUERY 1: Overall Complaint Trends Over Time
# ============================================================
print("\n" + "="*60)
print("Q1: How many complaints per month? (Trend)")
print("="*60)

query1 = """
SELECT 
    year,
    month,
    month_name,
    COUNT(*) as total_complaints
FROM complaints
GROUP BY year, month, month_name
ORDER BY year, month
"""
result1 = pd.read_sql_query(query1, conn)
print(result1)
result1.to_csv('sql_results/q1_monthly_trends.csv', index=False)
print("✅ Saved: sql_results/q1_monthly_trends.csv")

# ============================================================
# QUERY 2: Complaints by Category
# ============================================================
print("\n" + "="*60)
print("Q2: Which complaint types are most common?")
print("="*60)

query2 = """
SELECT 
    complaint_category,
    COUNT(*) as total_complaints,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM complaints), 2) as percentage
FROM complaints
GROUP BY complaint_category
ORDER BY total_complaints DESC
"""
result2 = pd.read_sql_query(query2, conn)
print(result2)
result2.to_csv('sql_results/q2_complaint_categories.csv', index=False)
print("✅ Saved: sql_results/q2_complaint_categories.csv")

# ============================================================
# QUERY 3: Average Resolution Time by Category
# ============================================================
print("\n" + "="*60)
print("Q3: Which complaint types take longest to resolve?")
print("="*60)

query3 = """
SELECT 
    complaint_category,
    COUNT(*) as total_complaints,
    ROUND(AVG(resolution_time_hours), 2) as avg_resolution_hours,
    MIN(resolution_time_hours) as min_hours,
    MAX(resolution_time_hours) as max_hours
FROM complaints
WHERE resolution_time_hours IS NOT NULL
GROUP BY complaint_category
ORDER BY avg_resolution_hours DESC
"""
result3 = pd.read_sql_query(query3, conn)
print(result3)
result3.to_csv('sql_results/q3_resolution_by_category.csv', index=False)
print("✅ Saved: sql_results/q3_resolution_by_category.csv")

# ============================================================
# QUERY 4: Channel Performance
# ============================================================
print("\n" + "="*60)
print("Q4: Which channels resolve complaints fastest?")
print("="*60)

query4 = """
SELECT 
    channel,
    COUNT(*) as total_complaints,
    ROUND(AVG(resolution_time_hours), 2) as avg_resolution_hours,
    ROUND(AVG(satisfaction_score), 2) as avg_satisfaction
FROM complaints
WHERE resolution_time_hours IS NOT NULL
GROUP BY channel
ORDER BY avg_resolution_hours
"""
result4 = pd.read_sql_query(query4, conn)
print(result4)
result4.to_csv('sql_results/q4_channel_performance.csv', index=False)
print("✅ Saved: sql_results/q4_channel_performance.csv")

# ============================================================
# QUERY 5: Top Complained Products
# ============================================================
print("\n" + "="*60)
print("Q5: Which products get most complaints?")
print("="*60)

query5 = """
SELECT 
    product_name,
    product_category,
    COUNT(*) as complaint_count,
    ROUND(AVG(satisfaction_score), 2) as avg_satisfaction
FROM complaints
WHERE product_name != 'unknown'
GROUP BY product_name, product_category
ORDER BY complaint_count DESC
LIMIT 10
"""
result5 = pd.read_sql_query(query5, conn)
print(result5)
result5.to_csv('sql_results/q5_top_products.csv', index=False)
print("✅ Saved: sql_results/q5_top_products.csv")

# ============================================================
# QUERY 6: Store Performance
# ============================================================
print("\n" + "="*60)
print("Q6: Which stores have most complaints?")
print("="*60)

query6 = """
SELECT 
    store_location,
    COUNT(*) as total_complaints,
    ROUND(AVG(resolution_time_hours), 2) as avg_resolution_hours,
    ROUND(AVG(satisfaction_score), 2) as avg_satisfaction
FROM complaints
WHERE store_location NOT IN ('online', 'none', '')
    AND resolution_time_hours IS NOT NULL
GROUP BY store_location
ORDER BY total_complaints DESC
"""
result6 = pd.read_sql_query(query6, conn)
print(result6)
result6.to_csv('sql_results/q6_store_performance.csv', index=False)
print("✅ Saved: sql_results/q6_store_performance.csv")

# ============================================================
# QUERY 7: High-Value vs Regular Customer Comparison
# ============================================================
print("\n" + "="*60)
print("Q7: Do high-value customers get better service?")
print("="*60)

query7 = """
SELECT 
    is_high_value_customer,
    COUNT(*) as total_complaints,
    ROUND(AVG(resolution_time_hours), 2) as avg_resolution_hours,
    ROUND(AVG(satisfaction_score), 2) as avg_satisfaction,
    ROUND(SUM(CASE WHEN resolution_speed = 'fast' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_fast_resolution
FROM complaints
WHERE resolution_time_hours IS NOT NULL
GROUP BY is_high_value_customer
"""
result7 = pd.read_sql_query(query7, conn)
result7['is_high_value_customer'] = result7['is_high_value_customer'].map({0: 'Regular', 1: 'High-Value'})
print(result7)
result7.to_csv('sql_results/q7_customer_segment_comparison.csv', index=False)
print("✅ Saved: sql_results/q7_customer_segment_comparison.csv")

# ============================================================
# QUERY 8: Repeat Complainers Analysis
# ============================================================
print("\n" + "="*60)
print("Q8: How do repeat complaints differ from first-time?")
print("="*60)

query8 = """
SELECT 
    is_repeat_complaint,
    COUNT(*) as total_complaints,
    ROUND(AVG(resolution_time_hours), 2) as avg_resolution_hours,
    ROUND(AVG(satisfaction_score), 2) as avg_satisfaction
FROM complaints
WHERE resolution_time_hours IS NOT NULL
GROUP BY is_repeat_complaint
"""
result8 = pd.read_sql_query(query8, conn)
result8['is_repeat_complaint'] = result8['is_repeat_complaint'].map({0: 'First-Time', 1: 'Repeat'})
print(result8)
result8.to_csv('sql_results/q8_repeat_complaints.csv', index=False)
print("✅ Saved: sql_results/q8_repeat_complaints.csv")

# ============================================================
# QUERY 9: Resolution Status Breakdown
# ============================================================
print("\n" + "="*60)
print("Q9: What's the resolution status distribution?")
print("="*60)

query9 = """
SELECT 
    resolution_status,
    COUNT(*) as total_complaints,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM complaints), 2) as percentage,
    ROUND(AVG(satisfaction_score), 2) as avg_satisfaction
FROM complaints
GROUP BY resolution_status
ORDER BY total_complaints DESC
"""
result9 = pd.read_sql_query(query9, conn)
print(result9)
result9.to_csv('sql_results/q9_resolution_status.csv', index=False)
print("✅ Saved: sql_results/q9_resolution_status.csv")

# ============================================================
# QUERY 10: Day of Week Patterns
# ============================================================
print("\n" + "="*60)
print("Q10: Which days have most complaints?")
print("="*60)

query10 = """
SELECT 
    day_of_week,
    COUNT(*) as total_complaints,
    is_weekend
FROM complaints
GROUP BY day_of_week, is_weekend
ORDER BY 
    CASE day_of_week
        WHEN 'Monday' THEN 1
        WHEN 'Tuesday' THEN 2
        WHEN 'Wednesday' THEN 3
        WHEN 'Thursday' THEN 4
        WHEN 'Friday' THEN 5
        WHEN 'Saturday' THEN 6
        WHEN 'Sunday' THEN 7
    END
"""
result10 = pd.read_sql_query(query10, conn)
print(result10)
result10.to_csv('sql_results/q10_day_patterns.csv', index=False)
print("✅ Saved: sql_results/q10_day_patterns.csv")

# ============================================================
# QUERY 11: Holiday Season Impact
# ============================================================
print("\n" + "="*60)
print("Q11: How does holiday season affect complaints?")
print("="*60)

query11 = """
SELECT 
    is_holiday_season,
    COUNT(*) as total_complaints,
    ROUND(AVG(resolution_time_hours), 2) as avg_resolution_hours,
    complaint_category,
    COUNT(*) as category_count
FROM complaints
WHERE resolution_time_hours IS NOT NULL
GROUP BY is_holiday_season, complaint_category
ORDER BY is_holiday_season DESC, category_count DESC
"""
result11 = pd.read_sql_query(query11, conn)
result11['is_holiday_season'] = result11['is_holiday_season'].map({0: 'Regular Season', 1: 'Holiday Season'})
print(result11.head(15))
result11.to_csv('sql_results/q11_holiday_impact.csv', index=False)
print("✅ Saved: sql_results/q11_holiday_impact.csv")

# ============================================================
# QUERY 12: Satisfaction by Priority Level
# ============================================================
print("\n" + "="*60)
print("Q12: Does priority level affect satisfaction?")
print("="*60)

query12 = """
SELECT 
    priority_level,
    COUNT(*) as total_complaints,
    ROUND(AVG(satisfaction_score), 2) as avg_satisfaction,
    ROUND(AVG(resolution_time_hours), 2) as avg_resolution_hours
FROM complaints
WHERE satisfaction_score IS NOT NULL
    AND resolution_time_hours IS NOT NULL
GROUP BY priority_level
ORDER BY 
    CASE priority_level
        WHEN 'critical' THEN 1
        WHEN 'high' THEN 2
        WHEN 'medium' THEN 3
        WHEN 'low' THEN 4
    END
"""
result12 = pd.read_sql_query(query12, conn)
print(result12)
result12.to_csv('sql_results/q12_priority_satisfaction.csv', index=False)
print("✅ Saved: sql_results/q12_priority_satisfaction.csv")

# Close connection
conn.close()

print("\n" + "="*60)
print("✅ SQL ANALYSIS COMPLETE!")
print("="*60)
print("\nAll query results saved in 'sql_results/' folder")
print("Total queries executed: 12")


SQL ANALYSIS: BUSINESS QUESTIONS

Q1: How many complaints per month? (Trend)
    year  month month_name  total_complaints
0   2022      6       June              1044
1   2022      7       July              1053
2   2022      8     August              1067
3   2022      9  September              1054
4   2022     10    October              1073
5   2022     11   November               978
6   2022     12   December              1082
7   2023      1    January              1021
8   2023      2   February               944
9   2023      3      March              1064
10  2023      4      April              1032
11  2023      5        May              1076
12  2023      6       June              1020
13  2023      7       July              1081
14  2023      8     August              1071
15  2023      9  September              1051
16  2023     10    October              1073
17  2023     11   November              1023
18  2023     12   December              1007
19  2024      1    Janu